# Notebook 02 — Analisis Statistik Deskriptif & EDA
**Input**: `data/processed/demografi_clean.csv`  
**Tujuan**: Eksplorasi mendalam distribusi penduduk, tren pertumbuhan, uji hipotesis, dan analisis ketimpangan.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_csv('../data/processed/demografi_clean.csv')
print(f'Data loaded: {df.shape}')
df.head(3)

## 1. Statistik Deskriptif Lengkap

In [ ]:
target_cols = [
    'jumlah_penduduk_ribu_2026', 'kepadatan_per_km2_2026', 'laju_pertumbuhan_pct_2026',
    'laju_2020_2024', 'imr_per1000_2020', 'cbr_per1000_2020', 'tfr_2020', 'u5mr_per1000_2020'
]

def extended_describe(series):
    s = series.dropna()
    return pd.Series({
        'n'         : len(s),
        'mean'      : s.mean(),
        'median'    : s.median(),
        'std'       : s.std(),
        'cv (%)'    : s.std() / s.mean() * 100,
        'min'       : s.min(),
        'max'       : s.max(),
        'skewness'  : stats.skew(s),
        'kurtosis'  : stats.kurtosis(s),
        'sw_stat'   : stats.shapiro(s)[0],
        'sw_pval'   : stats.shapiro(s)[1]
    })

summary = pd.DataFrame({col: extended_describe(df[col]) for col in target_cols})
print('=== STATISTIK DESKRIPTIF EXTENDED ===')
summary.T

**Interpretasi**:
- `jumlah_penduduk` memiliki CV sangat tinggi (~164%) → distribusi sangat tidak merata
- `skewness > 2` pada penduduk & kepadatan → distribusi right-skewed kuat (dikuasai beberapa provinsi besar)
- Shapiro-Wilk `p < 0.05` pada hampir semua variabel → distribusi tidak normal, analisis nonparametrik lebih tepat

## 2. Koefisien Gini — Ketimpangan Distribusi Penduduk

In [ ]:
def gini(arr):
    """Hitung koefisien Gini (0 = merata sempurna, 1 = tidak merata total)"""
    a = np.sort(arr)
    n = len(a)
    return (2 * np.sum((np.arange(1, n+1) * a))) / (n * np.sum(a)) - (n+1)/n

g = gini(df['jumlah_penduduk_ribu_2026'].values)
print(f'Koefisien Gini distribusi penduduk antar provinsi: {g:.4f}')
print()
print('Interpretasi:')
print('  0.00 – 0.30  : Rendah (relatif merata)')
print('  0.30 – 0.50  : Sedang')
print('  0.50 – 1.00  : Tinggi (sangat tidak merata)')
print(f'  → Nilai {g:.3f} menunjukkan ketimpangan TINGGI')

# Konsentrasi Jawa
jawa_mask = df['pulau'] == 'Jawa'
jawa_pct = df[jawa_mask]['distribusi_pct_2026'].sum()
print(f'\nKonsentrasi penduduk Pulau Jawa : {jawa_pct:.1f}% (6 dari 38 provinsi)')

## 3. Tren Laju Pertumbuhan Historis

In [ ]:
periode_cols = [
    ('1971–1980', 'laju_1971_1980'),
    ('1980–1990', 'laju_1980_1990'),
    ('1990–2000', 'laju_1990_2000'),
    ('2000–2010', 'laju_2000_2010'),
    ('2010–2020', 'laju_2010_2020'),
    ('2020–2024', 'laju_2020_2024'),
]

print('=== TREN LAJU PERTUMBUHAN (median nasional, %/tahun) ===')
trend = []
for label, col in periode_cols:
    vals = df[col].dropna()
    trend.append({'periode': label, 'median': vals.median(), 'mean': vals.mean(), 'n': len(vals)})
    
trend_df = pd.DataFrame(trend)
print(trend_df.to_string(index=False))
print()
print('→ Tren umum MENURUN — perlambatan transisi demografi')
print('→ Anomali kenaikan 2000–2010: otonomi daerah & dekonsentrasi penduduk')

## 4. Uji-T: Laju Pertumbuhan Jawa vs Luar Jawa

In [ ]:
jawa_laju = df[df['pulau'] == 'Jawa']['laju_2020_2024'].dropna()
luar_laju = df[df['pulau'] != 'Jawa']['laju_2020_2024'].dropna()

# Levene test untuk kesamaan varians
levene_stat, levene_p = stats.levene(jawa_laju, luar_laju)
equal_var = levene_p > 0.05

# Independent t-test
t_stat, t_p = stats.ttest_ind(jawa_laju, luar_laju, equal_var=equal_var)

# Effect size (Cohen's d)
pooled_std = np.sqrt((jawa_laju.std()**2 + luar_laju.std()**2) / 2)
cohens_d = (jawa_laju.mean() - luar_laju.mean()) / pooled_std

print('=== UJI-T INDEPENDEN: Laju Pertumbuhan Jawa vs Luar Jawa (2020–2024) ===')
print(f'Levene test         : stat={levene_stat:.4f}, p={levene_p:.4f} → equal_var={equal_var}')
print(f'Jawa    (n={len(jawa_laju)})   : mean={jawa_laju.mean():.4f}%, std={jawa_laju.std():.4f}')
print(f'Luar Jawa (n={len(luar_laju)}) : mean={luar_laju.mean():.4f}%, std={luar_laju.std():.4f}')
print(f't-statistic         : {t_stat:.4f}')
print(f'p-value             : {t_p:.6f}')
print(f"Cohen's d           : {cohens_d:.4f}")
print()
sig = 'SIGNIFIKAN' if t_p < 0.05 else 'TIDAK SIGNIFIKAN'
print(f'→ Kesimpulan: Perbedaan {sig} (α = 0.05)')
print('→ Luar Jawa tumbuh lebih cepat, konsisten dengan redistribusi penduduk & DOB')

## 5. Analisis Korelasi Antar Variabel Demografis

In [ ]:
corr_cols = [
    'kepadatan_per_km2_2026', 'laju_2020_2024',
    'imr_per1000_2020', 'cbr_per1000_2020', 'tfr_2020', 'u5mr_per1000_2020'
]

corr_df = df[corr_cols].dropna()

# Pearson
print('=== MATRIKS KORELASI PEARSON ===')
print(corr_df.corr().round(3))

print()
# Spearman (lebih robust untuk distribusi skewed)
print('=== MATRIKS KORELASI SPEARMAN (lebih robust) ===')
print(corr_df.corr(method='spearman').round(3))

print()
print('Temuan utama:')
print('  CBR ↔ TFR    : r ≈ 0.96 (sangat kuat) — angka kelahiran kasar erat dengan total fertilitas')
print('  IMR ↔ TFR    : r ≈ 0.81 (kuat) — wilayah IMR tinggi = TFR tinggi (transisi demografi)')
print('  Kepadatan ↔ TFR : r ≈ -0.51 (sedang-negatif) — provinsi padat cenderung TFR lebih rendah')

## 6. Ranking Provinsi — Top & Bottom Performers

In [ ]:
print('=== TOP 5 PERTUMBUHAN TERTINGGI (2020-2024) ===')
print(df.nlargest(5, 'laju_2020_2024')[['provinsi','laju_2020_2024','pulau']].to_string(index=False))

print()
print('=== TOP 5 PERTUMBUHAN TERENDAH (2020-2024) ===')
print(df.nsmallest(5, 'laju_2020_2024')[['provinsi','laju_2020_2024','pulau']].to_string(index=False))

print()
print('=== IMR TERBAIK (terendah) ===')
print(df.nsmallest(5, 'imr_per1000_2020')[['provinsi','imr_per1000_2020','tfr_2020']].to_string(index=False))

print()
print('=== IMR TERBURUK (tertinggi) ===')
print(df.nlargest(5, 'imr_per1000_2020')[['provinsi','imr_per1000_2020','tfr_2020']].to_string(index=False))

## 7. Analisis Ketimpangan Per Pulau

In [ ]:
pulau_stats = df.groupby('pulau').agg(
    n_provinsi=('provinsi', 'count'),
    total_penduduk_juta=('jumlah_penduduk_ribu_2026', lambda x: x.sum()/1000),
    distribusi_pct=('distribusi_pct_2026', 'sum'),
    median_laju=('laju_2020_2024', 'median'),
    median_imr=('imr_per1000_2020', 'median'),
    median_tfr=('tfr_2020', 'median')
).round(2).sort_values('total_penduduk_juta', ascending=False)

print('=== RINGKASAN PER PULAU ===')
print(pulau_stats.to_string())